# Lab08.1 — Understanding the GTSRB Dataset

**Objective:** explore genuine GTSRB images, the 43 official labels, class distribution, and RGB image preprocessing.

**Hardware:** Digilent Zybo Legacy, PYNQ 3.0.1; no FPGA accelerator needed. It works with the official CSV and at least one photograph (`00105.ppm`).

**Required files:** `Lab08/data/gtsrb/GT-final_test.csv` and `Lab08/data/gtsrb/GTSRB/Final_Test/Images/00105.ppm`. See `Lab08/README.md` for instructions. No full image archive is needed on Zybo.


## 1 — Dataset and ground-truth labels

The official CSV contains the image filename and correct category. A category is an integer from 0 through 42. This notebook reads the CSV **without scanning 12,630 files**, so the Zybo Jupyter file browser remains responsive.


In [ ]:
from pathlib import Path
import csv
import sys
import numpy as np
import cv2
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()         # notebook located in Lab08/notebooks
TOOLS = ROOT / 'tools'
if str(TOOLS) not in sys.path:
    sys.path.insert(0, str(TOOLS))
from gtsrb import LABELS

DATA = ROOT / 'data' / 'gtsrb'
CSV_FILE = DATA / 'GT-final_test.csv'
IMAGE_DIR = DATA / 'GTSRB' / 'Final_Test' / 'Images'
if not CSV_FILE.is_file():
    raise FileNotFoundError(f'Missing official GTSRB ground truth: {CSV_FILE}')

with CSV_FILE.open('r', newline='', encoding='utf-8-sig') as f:
    records = list(csv.DictReader(f, delimiter=';', skipinitialspace=True))
if not records or not {'Filename', 'ClassId'}.issubset(records[0]):
    raise ValueError('Unexpected GTSRB CSV columns')
labels = np.array([int(row['ClassId']) for row in records], dtype=np.int64)
if np.any((labels < 0) | (labels >= 43)):
    raise ValueError('ClassId outside the 43 GTSRB categories')
print('Labeled records:', len(records))
print('Number of classes:', len(LABELS))
print('Available image directory:', IMAGE_DIR)
print('Example:', records[105]['Filename'], '->', records[105]['ClassId'])


## 2 — Distribution of the 43 categories

Class frequencies are calculated from **all CSV rows**, not from the photographs currently stored on the board. The number of stored images may be much smaller than the number of labeled records.


In [ ]:
counts = np.bincount(np.asarray(labels, dtype=np.int32), minlength=43)
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(np.arange(43), counts)
ax.set(xlabel='GTSRB class ID', ylabel='Number of test images',
       title='Official GTSRB test-set distribution')
ax.set_xticks(np.arange(43))
ax.tick_params(axis='x', labelsize=7)
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()
print('Largest class:', int(np.argmax(counts)), LABELS[int(np.argmax(counts))], int(counts.max()))
print('Smallest class:', int(np.argmin(counts)), LABELS[int(np.argmin(counts))], int(counts.min()))


## 3 — Inspect real photographs

With the complete dataset, display 12 reproducibly selected images. On Zybo, use the known small set of transferred images, including `00105.ppm`. Missing files are skipped rather than triggering a scan of all image files.


In [ ]:
# Inspect selected CSV entries only; never call rglob() over the whole image archive.
rng = np.random.default_rng(2026)
if (IMAGE_DIR / records[0]['Filename']).is_file():
    selected = rng.choice(len(records), size=min(12, len(records)), replace=False).tolist()
else:
    selected = [105]             # one sample supplied for the Zybo laboratory

available = []
for idx in selected:
    item = records[int(idx)]
    path = IMAGE_DIR / Path(item['Filename']).name
    if path.is_file():
        available.append((path, int(item['ClassId'])))

if not available:
    print('No selected image files available. Check IMAGE_DIR and copy 00105.ppm if needed.')
else:

    ncols = min(4, len(available))
    nrows = (len(available) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(3 * ncols, 3 * nrows), squeeze=False)
    for ax in axes.flat:
        ax.axis('off')
    for ax, (path, class_id) in zip(axes.flat, available):
        image_bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
        if image_bgr is None:
            raise IOError(f'Cannot decode {path}')
        ax.imshow(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
        ax.set_title(f'{class_id}: {LABELS[class_id]}', fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    plt.show()
print('Photographs displayed:', len(available))


## 4 — Preprocessing: RGB and 32 × 32 pixels

This exercise demonstrates channel order and a direct `32 × 32` resize (3,072 RGB bytes). OpenCV uses BGR by default. **This is not the final input preparation for the pretrained BNN in Lab08.2.** That model requires a CIFAR-style record (3,073 bytes including a label byte) with aspect-ratio-preserving resize and padding. Compare the two methods.


In [ ]:
example_path = IMAGE_DIR / '00105.ppm'
if not example_path.is_file():
    if not available:
        raise FileNotFoundError('No photograph available for preprocessing demonstration')
    example_path = available[0][0]
photo_bgr = cv2.imread(str(example_path), cv2.IMREAD_COLOR)
if photo_bgr is None:
    raise IOError(f'Cannot read {example_path}')
photo_rgb = cv2.cvtColor(photo_bgr, cv2.COLOR_BGR2RGB)
rgb32 = cv2.cvtColor(cv2.resize(photo_bgr, (32, 32),
                                interpolation=cv2.INTER_LINEAR), cv2.COLOR_BGR2RGB)
assert rgb32.shape == (32, 32, 3) and rgb32.dtype == np.uint8
assert rgb32.nbytes == 3072
fig, axes = plt.subplots(1, 2, figsize=(7, 3))
axes[0].imshow(photo_rgb)
axes[0].set_title(f'Original: {photo_rgb.shape[1]} × {photo_rgb.shape[0]}')
axes[1].imshow(rgb32)
axes[1].set_title('Basic RGB resize: 32 × 32')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()
print('File:', example_path.name, '| Array:', rgb32.shape, rgb32.dtype, '| Bytes:', rgb32.nbytes)


## Exercises

1. Identify the most and least frequent GTSRB categories from the CSV histogram. Why does class imbalance matter?
2. Compare the original `00105.ppm` photograph with the 32 × 32 RGB resize. Which details are lost?
3. Explain why a `32 × 32 × 3` RGB image occupies 3,072 bytes. Why does the CIFAR-style record in Lab08.2 have 3,073 bytes?
4. Compare direct resizing here with the aspect-ratio-preserving preparation in Lab08.2.

**Next:** Lab08.2 uses a pretrained CNV-W1A1 network and software ARM inference (no FPGA overlay); FPGA acceleration is planned for a future lab.
